# 06 — MEM Electricity | Automatic Download (raw landing)

Finds the newest **Informe de Desempeño — Anexos** workbook published by the Ministry of Energy and Mines (MEM)
and lands it, unchanged, in the raw archive `/Volumes/rd_economic_intelligence/bronze/mem_raw/ede_performance/`.

| Step | Rule |
|---|---|
| Discover | MEM website media index (WordPress REST API `wp-json/wp/v2/media`), files whose name contains `Desempeno` and `Anexos` and end in `.xlsx`; newest by publication date |
| Skip | The file is downloaded and its SHA-256 compared with every file already archived; identical content is never landed twice |
| Validate | The download must open as a workbook with the sheets `EDE's` and `Nuevo Regimen tarifario`; otherwise the task fails and nothing is written |
| Land | Append-only archive, one folder per arrival month: `ede_performance/<yyyy>/<mm>/<original name>`. Files are never overwritten or deleted. |
| Corrections | If MEM republishes a file under a name already archived but with different content, it is kept as a new version: `<name>__<yyyy-mm-dd>.xlsx` |
| Signal | Task value `downloaded` (true/false) lets the scheduled Job skip Bronze, Silver and Gold when nothing new was published |

Bronze (`06_MEM_Electricity_Bronze`) picks up new archive files incrementally with Auto Loader. **Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import json, hashlib, io, os, re, urllib.request
from datetime import datetime
from zoneinfo import ZoneInfo
import openpyxl

LANDING = "/Volumes/rd_economic_intelligence/bronze/mem_raw/ede_performance/"
API = "https://mem.gob.do/wp-json/wp/v2/media?search=Anexos&per_page=100&orderby=date&order=desc&_fields=date,source_url"
REQUIRED_SHEETS = {"EDE's", "Nuevo Regimen tarifario"}
HEADERS = {"User-Agent": "rd-economic-intelligence/1.0 (portfolio data pipeline)"}
NOW = datetime.now(ZoneInfo("America/Santo_Domingo"))

def get(url, timeout=60):
    with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=timeout) as r:
        return r.read()

def flag(downloaded):
    # Read by the scheduled Job's condition task: downstream layers run only when a new file landed
    dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

items = json.loads(get(API))
cands = [(i["date"], i["source_url"]) for i in items
         if re.search(r"desempe(n|ñ)o", i["source_url"], re.I) and "anexo" in i["source_url"].lower() and i["source_url"].lower().endswith(".xlsx")]
if not cands:
    raise RuntimeError("No MEM annex workbook found in the media index")
pub_date, url = max(cands)
name = url.rsplit("/", 1)[-1]
print(f"Newest annex: {name} (published {pub_date[:10]})")

# Every workbook already in the archive (all month folders; folders starting with '_' are ignored)
archived = []
for root, dirs, files in os.walk(LANDING):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    archived += [os.path.join(root, f) for f in files if f.lower().endswith(".xlsx")]

data = get(url, timeout=180)
digest = hashlib.sha256(data).hexdigest()
for p in archived:
    if os.path.getsize(p) == len(data) and sha256_file(p) == digest:
        flag(False)
        dbutils.notebook.exit(json.dumps({"status": "SKIPPED", "reason": f"same content already archived as {p[len(LANDING):]}",
                                          "file": name, "published": pub_date[:10]}))

wb = openpyxl.load_workbook(io.BytesIO(data), read_only=True)
missing = REQUIRED_SHEETS - set(wb.sheetnames)
if missing:
    raise ValueError(f"Downloaded file failed the structural contract, missing sheets: {sorted(missing)}")

stem, ext = name.rsplit(".", 1)
is_correction = any(os.path.basename(p) == name for p in archived)
out_name = f"{stem}__{NOW:%Y-%m-%d}.{ext}" if is_correction else name
target_dir = f"{LANDING}{NOW:%Y}/{NOW:%m}/"
os.makedirs(target_dir, exist_ok=True)
if os.path.exists(target_dir + out_name):  # never overwrite: same name twice on the same day
    out_name = f"{stem}__{NOW:%Y-%m-%d_%H%M%S}.{ext}"
with open(target_dir + out_name, "wb") as fh:
    fh.write(data)
flag(True)
dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED", "path": target_dir[len(LANDING):] + out_name, "correction_of_archived_file": is_correction,
                                  "published": pub_date[:10], "bytes": len(data), "sha256": digest[:16]}))